# SOLT 校准

这个简单的示例演示了如何使用 scikit-rf 进行基本的 SOLT 校准。

## 基本用法### 导入首先，我们导入 `skrf` 库、`SOLT` 类，并设置 matplotlib 绘图环境。

In [ ]:
import skrf as rf
from skrf.network import two_port_reflect

rf.stylely()

### 典型的 SOLT 校准流程

双端口校准的执行方式与单端口校准相同。但是，scikit-rf 仅接受双端口网络的测量结果，即使是对于反射和负载校准标准也是如此。

#### 分别测量两个校准标准如果您没有两组校准标准，也没关系！您可以分别测量每个端口上的反射标准。例如，将一个短路连接到端口 1，将单端口测量结果保存为 `short.s1p`，然后将相同的短路移动到端口 2，将另一个单端口测量结果保存为 `short2.s1p` 或类似的文件名。接下来，您可以使用函数 `skrf.network.two_port_reflect` 从两个单端口网络创建两个端口的网络：```pythonshort = rf.Network('ideals/short.s1p')  # 一个 1 端口网络shorts = two_port_reflect(short, short)  # 一个 2 端口网络```函数 `skrf.network.two_port_reflect` 的作用如下：<img src="2_1port_to_1_2port.svg" width="50%"/>#### 同时测量两个校准标准通常，您可以同时测量两个反射标准，然后将数据直接保存为两个端口的网络。例如，将第一个短路连接到端口 1，将第二个短路连接到端口 2，并将所有内容保存为两个端口的测量结果，例如 `short,short.s2p` 或类似的文件名。<img src="VNA_2_1port.svg" width="30%"/>#### 隔离校准对于开路、短路和负载校准，即使将输入作为具有四个 S 散射参数的两个端口网络提供给 scikit-rf，也会忽略 $S_{21}$ 和 $S_{12}$——根据定义，两者都等于 0。仅使用 $S_{11}$ 和 $S_{22}$，即使对于负载-负载测量也是如此。因此，默认情况下不会进行隔离校准。如果需要隔离校准，则必须明确告诉 scikit-rf 执行此操作。具体来说，`SOLT()` 类接受一个可选参数 `isolation`，用于此目的。要校准隔离，请将此参数设置为一个两个端口的网络，该网络代表在两个端口上都连接负载的测量结果。

### 代码典型的 SOLT 校准流程如下：

```chinese# 包含“理想”响应的网络类型列表my_ideals = [    rf.Network('ideal/short, short.s2p'),    rf.Network('ideal/open, open.s2p'),    rf.Network('ideal/load, load.s2p'),    rf.Network('ideal/thru.s2p'),]# 包含“测量”响应的网络类型列表my_measured = [    rf.Network('measured/short, short.s2p'),    rf.Network('measured/open, open.s2p'),    rf.Network('measured/load, load.s2p'),    rf.Network('measured/thru.s2p'),]## 创建 SOLT 实例cal = SOLT(    ideals = my_ideals,    measured = my_measured,    isolation = rf.Network('measured/load, load.s2p'),    # 隔离校准是可选的，可以删除。)## 运行，并将校准应用于 DUT# 运行校准算法cal.run()# 将其应用于 DUTdut = rf.Network('my_dut.s2p')dut_caled = cal.apply_cal(dut)# 绘制结果dut_caled.plot_s_db()# 保存结果dut_caled.write_touchstone()```

## 示例以下示例说明了一种常见的情况：DUT（被测设备）通过两条不同长度的电缆连接到 VNA。校准的目的是将参考平面移动到 DUT，即从测量结果中消除电缆的影响。<img src="line1_dut_line2.svg" width="60%"/>

在下面的示例中，DUT（测试对象）是已知的，只是为了验证校准方法是否有效。当然，在实际应用中，DUT通常是未知的……

In [ ]:
dut = rf.data.ring_slot
dut.plot_s_db(lw=2)  # this is what we should find after the calibration

理想的元件网络可以从您的校准套件制造商处获得，也可以通过建模获得。在这个例子中，我们模拟来自传输线理论的理想元件。我们创建了一条有损耗和噪声的传输线（为了便于说明）。

In [ ]:
media = rf.media.DefinedGammaZ0(frequency=dut.frequency, gamma=0.5 + 1j)

然后，我们创建理想的元件：短路、开路和负载，以及直通。默认情况下，`media.short()`、`media.open()` 和 `media.match()` 方法返回一个单端口网络。SOLT 类需要一个由两个双端口网络组成的列表，因此需要使用 `two_port_reflect()` 方法，从两个单端口网络中创建一个双端口网络（`media.thru()` 返回一个双端口网络，无需进行调整）。或者，可以使用 `nports=2` 参数来简化此操作。

In [ ]:
# ideal 1-port Networks
short_ideal = media.short()
open_ideal = media.open()
load_ideal = media.match()  # could also be: media.load(Gamma0=0)
thru_ideal = media.thru()

# forge a two-port network from two one-port networks
short_ideal_2p = two_port_reflect(short_ideal, short_ideal)
open_ideal_2p = two_port_reflect(open_ideal, open_ideal)
load_ideal_2p = two_port_reflect(load_ideal, load_ideal)

# alternatively, the "nport=2" argument can be used as a shorthand
# short_ideal_2p = media.short(nports=2)
# open_ideal_2p = media.open(nports=2)
# load_ideal_2p = media.match(nports=2)

现在我们已经有了理想元件，让我们模拟测量过程。请注意，在下面的示例中，传输线不是对称的，这是为了使其尽可能通用。在这种情况下，需要调用 [flipped()](../../api/generated/skrf.network.Network.flipped.rst) 方法，以便将理想元件连接到 `line2` 对象的正确侧。

In [ ]:
# left and right piece of transmission lines
line1 = media.line(d=20, unit='cm')**media.impedance_mismatch(1,2)
line2 = media.line(d=30, unit='cm')**media.impedance_mismatch(1,3)

# add some noise to make it more realistic
line1.add_noise_polar(.01, .1)
line2.add_noise_polar(.01, .1)

# fake the measured setup
measured = line1 ** dut  ** line2

# fake the calibration measurements
# Note the use of flipped() on line2
open_measured = two_port_reflect(line1 ** media.open(), line2.flipped() ** media.open())
short_measured = two_port_reflect(line1 ** media.short(), line2.flipped() ** media.short())
load_measured = two_port_reflect(line1 ** media.load(Gamma0=0), line2.flipped() ** media.load(Gamma0=0))
thru_measured = line1 ** line2

现在我们可以创建 `SOLT` 类所期望的 `Network` 对象列表：

In [ ]:
# a list of Network types, holding 'ideal' responses
my_ideals = [
    short_ideal_2p,
    open_ideal_2p,
    load_ideal_2p,
    thru_ideal,   # Thru should be the last
    ]

# a list of Network types, holding 'measured' responses
my_measured = [
    short_measured,
    open_measured,
    load_measured,
    thru_measured,   # Thru should be the last
]

## create a SOLT instance
cal = rf.calibration.SOLT(
    ideals = my_ideals,
    measured = my_measured,
    )

最后，应用校准：

In [ ]:
# run calibration algorithm
cal.run()

# apply it to a dut
measured_caled = cal.apply_cal(measured)

让我们看一下 S11 和 S21 的结果：

In [ ]:
measured.plot_s_db(m=0, n=0, lw=2, label='measured')
measured_caled.plot_s_db(m=0, n=0, lw=2, label='caled')
dut.plot_s_db(m=0, n=0, ls='--', lw=2, label='expected')

In [ ]:
measured.plot_s_db(m=1, n=0, lw=2, label='measured')
measured_caled.plot_s_db(m=1, n=0, lw=2, label='caled')
dut.plot_s_db(m=1, n=0, ls='--', lw=2, label='expected')

经过校准的网络（在很大程度上）与预期的待测设备（DUT）相同：

In [ ]:
dut == measured

In [ ]:
dut == measured_caled  # within 1e-4 absolute tolerance